In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 0 — KEEP COLAB ALIVE
# ══════════════════════════════════════════════════════════════

from IPython.display import Javascript
from IPython.display import display


def keep_colab_alive():

    display(Javascript('''
    function ClickConnect(){
        console.log("Keeping Colab alive");
        document.querySelector("colab-toolbar-button#connect").click()
    }
    setInterval(ClickConnect, 60000)
    '''))


keep_colab_alive()

<IPython.core.display.Javascript object>

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — INSTALL DEPENDENCIES
# ══════════════════════════════════════════════════════════════

!pip install -q \
    segmentation-models-pytorch \
    albumentations \
    timm \
    rasterio \
    tqdm \
    mlflow \
    huggingface_hub \
    opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — IMPORTS + DRIVE MOUNT
# ══════════════════════════════════════════════════════════════

!pip install rasterio
!pip install segmentation_models_pytorch
!pip install albumentations

from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import json
import warnings
import zipfile
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pathlib import Path
from tqdm.auto import tqdm
from scipy.ndimage import uniform_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import segmentation_models_pytorch as smp
import albumentations as A

from huggingface_hub import (
    hf_hub_download,
    list_repo_files
)

warnings.filterwarnings('ignore')

BASE_DIR = '/content/drive/MyDrive/DAI4'

RAW_DIR = f'{BASE_DIR}/data/raw'
PROC_DIR = f'{BASE_DIR}/data/processed'
CHECKPOINT_DIR = f'{BASE_DIR}/models/checkpoints'
RESULTS_DIR = f'{BASE_DIR}/results'

for d in [
    RAW_DIR,
    PROC_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR
]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('Device:', DEVICE)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Mounted at /content/drive
Device: cpu


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — GLOBAL VARIABLES
# ══════════════════════════════════════════════════════════════

LABEL_MAPPING = None

TARGET_HW = 512
TILE_SIZE = 256
OVERLAP = 32
MIN_DAMAGE = 0.02

N_CHANNELS = 9
NUM_CLASSES = 4

BATCH_SIZE = 4
NUM_WORKERS = 2

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — DOWNLOAD DATASET
# ══════════════════════════════════════════════════════════════

REPO_ID = 'Kullervo/BRIGHT'
REPO_TYPE = 'dataset'

print('Available files:')

files = list_repo_files(
    REPO_ID,
    repo_type=REPO_TYPE
)

for f in files:
    print(f)

Available files:
.gitattributes
README.md
aoi_reference_kmz.zip
cvprw26_test.zip
cvprw26_trainval_instance_labels.zip
dfc25_track2_test.zip
dfc25_track2_test_labels.zip
dfc25_track2_test_new.zip
dfc25_track2_trainval.zip
dfc25_track2_val_labels.zip
overall.jpg
post-event.zip
pre-event.zip
target.zip
umim_bata_explosion.zip
umim_noto_earthquake.zip
umim_spain_volcano.zip


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — DOWNLOAD + EXTRACT
# ══════════════════════════════════════════════════════════════


def download_and_extract(filename):

    extract_folder = filename.replace('.zip', '')

    extract_path = os.path.join(
        RAW_DIR,
        extract_folder
    )

    if os.path.exists(extract_path):
        print(f'{filename} already extracted')
        return

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type=REPO_TYPE,
        local_dir='/content'
    )

    with zipfile.ZipFile(downloaded, 'r') as zf:

        members = zf.infolist()

        for member in tqdm(
            members,
            desc=f'Extracting {filename}'
        ):
            zf.extract(member, RAW_DIR)

    print(f'Finished: {filename}')


for z in [
    'dfc25_track2_trainval.zip',
    'dfc25_track2_val_labels.zip',
    'dfc25_track2_test.zip'
]:
    download_and_extract(z)

dfc25_track2_trainval.zip already extracted
dfc25_track2_val_labels.zip already extracted


dfc25_track2_test.zip:   0%|          | 0.00/407M [00:00<?, ?B/s]

Extracting dfc25_track2_test.zip:   0%|          | 0/315 [00:00<?, ?it/s]

Finished: dfc25_track2_test.zip


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — DATASET PATHS
# ══════════════════════════════════════════════════════════════

TRAIN_PRE_DIR = f'{RAW_DIR}/dfc25_track2_trainval/train/pre-event'
TRAIN_POST_DIR = f'{RAW_DIR}/dfc25_track2_trainval/train/post-event'
TRAIN_LBL_DIR = f'{RAW_DIR}/dfc25_track2_trainval/train/target'

VAL_PRE_DIR = f'{RAW_DIR}/dfc25_track2_trainval/val/pre-event'
VAL_POST_DIR = f'{RAW_DIR}/dfc25_track2_trainval/val/post-event'
VAL_LBL_DIR = f'{RAW_DIR}/dfc25_track2_val_labels/target'

TEST_PRE_DIR = f'{RAW_DIR}/dfc25_track2_test/test/pre-event'
TEST_POST_DIR = f'{RAW_DIR}/dfc25_track2_test/test/post-event'

for p in [
    TRAIN_PRE_DIR,
    TRAIN_POST_DIR,
    TRAIN_LBL_DIR,
    VAL_PRE_DIR,
    VAL_POST_DIR,
    VAL_LBL_DIR
]:
    print(p, os.path.exists(p))

/content/drive/MyDrive/DAI4/data/raw/dfc25_track2_trainval/train/pre-event True
/content/drive/MyDrive/DAI4/data/raw/dfc25_track2_trainval/train/post-event True
/content/drive/MyDrive/DAI4/data/raw/dfc25_track2_trainval/train/target True
/content/drive/MyDrive/DAI4/data/raw/dfc25_track2_trainval/val/pre-event True
/content/drive/MyDrive/DAI4/data/raw/dfc25_track2_trainval/val/post-event True
/content/drive/MyDrive/DAI4/data/raw/dfc25_track2_val_labels/target True


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — BUILD TRIPLETS
# ══════════════════════════════════════════════════════════════


def get_tif_files(folder):
    return sorted(Path(folder).glob('*.tif'))


def extract_id(filepath):

    stem = Path(filepath).stem

    for suffix in [
        '_pre_disaster',
        '_post_disaster',
        '_building_damage',
        '_pre_event',
        '_post_event',
        '_label',
        '_mask'
    ]:
        stem = stem.replace(suffix, '')

    return stem


def build_triplets(pre_dir, post_dir, lbl_dir):

    pre_files = get_tif_files(pre_dir)
    post_files = get_tif_files(post_dir)
    lbl_files = get_tif_files(lbl_dir)

    pre_dict = {
        extract_id(f): f
        for f in pre_files
    }

    post_dict = {
        extract_id(f): f
        for f in post_files
    }

    lbl_dict = {
        extract_id(f): f
        for f in lbl_files
    }

    ids = sorted(
        set(pre_dict) &
        set(post_dict) &
        set(lbl_dict)
    )
    triplets = []

    for sid in ids:

        triplets.append(
            (
                sid,
                pre_dict[sid],
                post_dict[sid],
                lbl_dict[sid]
            )
        )

    return triplets


train_triplets = build_triplets(
    TRAIN_PRE_DIR,
    TRAIN_POST_DIR,
    TRAIN_LBL_DIR
)

val_triplets = build_triplets(
    VAL_PRE_DIR,
    VAL_POST_DIR,
    VAL_LBL_DIR
)

print('Train:', len(train_triplets))
print('Val:', len(val_triplets))

Train: 2890
Val: 349


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — PREPROCESSING FUNCTIONS
# ══════════════════════════════════════════════════════════════


def load_tif(path, target_hw=None):

    with rasterio.open(path) as src:
        arr = src.read()

    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]

    elif arr.ndim == 3:
        arr = arr.transpose(1, 2, 0)

    if target_hw:

        interp = cv2.INTER_NEAREST \
            if arr.ndim == 2 \
            else cv2.INTER_LINEAR

        arr = cv2.resize(
            arr.astype(np.float32),
            (target_hw, target_hw),
            interpolation=interp
        )

    return arr.astype(np.float32)



def normalize_channel(ch):

    mn = np.nanmin(ch)
    mx = np.nanmax(ch)

    return (ch - mn) / (mx - mn + 1e-8)
def compute_ndvi(red, nir):

    d = red + nir

    return np.where(
        d > 0,
        (nir - red) / d,
        0.0
    )



def compute_ndwi(green, nir):

    d = green + nir

    return np.where(
        d > 0,
        (green - nir) / d,
        0.0
    )



def compute_sar_coherence(vv, vh, window=5):

    a = vv.astype(np.complex64)
    b = vh.astype(np.complex64)

    cross = uniform_filter(
        np.abs(a * np.conj(b)),
        size=window
    )

    pa = uniform_filter(
        np.real(a * np.conj(a)),
        size=window
    )

    pb = uniform_filter(
        np.real(b * np.conj(b)),
        size=window
    )

    denom = np.sqrt(pa * pb)

    coh = np.where(denom > 0,
        cross / denom,
        0.0
    )

    return np.clip(np.abs(coh), 0.0, 1.0)



def preprocess_triplet(pre_path, post_path, lbl_path):

    opt = load_tif(pre_path, TARGET_HW)

    if opt.ndim == 2:
        opt = np.stack([opt] * 3, axis=-1)

    opt = opt[:, :, :3]

    sar = load_tif(post_path, TARGET_HW)

    if sar.ndim == 2:
        sar = np.stack([sar] * 2, axis=-1)

    sar = sar[:, :, :2]

    lbl = load_tif(lbl_path, TARGET_HW).astype(np.int64)

    if lbl.ndim == 3:
        lbl = lbl[:, :, 0]

    lbl = np.clip(lbl, 0, 3)

    opt_norm = np.stack([
        normalize_channel(opt[:, :, c])
        for c in range(3)
    ], axis=-1)

    sar_norm = np.stack([
        normalize_channel(sar[:, :, c])
        for c in range(2)
    ], axis=-1)

    R = opt_norm[:, :, 0]
    G = opt_norm[:, :, 1]
    B = opt_norm[:, :, 2]

    nir_approx = 0.4 * R + 0.3 * G + 0.3 * B

    ndvi = (compute_ndvi(R, nir_approx) + 1) / 2

    ndwi = (compute_ndwi(G, nir_approx) + 1) / 2

    pseudo_nbr = np.abs(
        nir_approx - sar_norm[:, :, 0]
    )
    pseudo_nbr = normalize_channel(pseudo_nbr)

    coh = compute_sar_coherence(
        sar_norm[:, :, 0],
        sar_norm[:, :, 1]
    )

    stacked = np.concatenate([
        opt_norm,
        sar_norm,
        ndvi[:, :, None],
        ndwi[:, :, None],
        pseudo_nbr[:, :, None],
        coh[:, :, None]
    ], axis=-1)

    return stacked.astype(np.float32), lbl.astype(np.int64)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — TILE GENERATION
# ══════════════════════════════════════════════════════════════


def tile_and_save(
    stacked,
    label,
    sid,
    tiles_dir,
    labels_dir,
    tile_size=256,
    overlap=32,
    min_damage=0.02
):

    stride = tile_size - overlap

    H, W = label.shape

    saved = 0

    for y in range(0, H - tile_size + 1, stride):

        for x in range(0, W - tile_size + 1, stride):

            tile = stacked[
                y:y+tile_size,
                x:x+tile_size
            ]

            lbl = label[
                y:y+tile_size,
                x:x+tile_size
            ]

            if np.sum(lbl > 0) / (tile_size**2) > min_damage:

                file_id = f'{sid}_{y}_{x}'

                np.save(
                    tiles_dir / f'{file_id}.npy',
                    tile
                )

                np.save(
                    labels_dir / f'{file_id}.npy',
                    lbl
                )

                saved += 1

    return saved

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — SMART PREPROCESSING
# ══════════════════════════════════════════════════════════════

def preprocess_split(triplets, split_name):

    tiles_dir = Path(PROC_DIR) / split_name / 'tiles'

    labels_dir = Path(PROC_DIR) / split_name / 'labels'

    tiles_dir.mkdir(parents=True, exist_ok=True)

    labels_dir.mkdir(parents=True, exist_ok=True)

    total_saved = 0

    for sid, pre, post, lbl in tqdm(
        triplets,
        desc=f'Preprocessing {split_name}'
    ):

        # ─────────────────────────────────────────────────────
        # Resume Logic
        # Skip ONLY if this sample already processed
        # ─────────────────────────────────────────────────────
        existing = list(
            tiles_dir.glob(f'{sid}_*.npy')
        )

        if len(existing) > 0:
            continue

        try:

            stacked, label = preprocess_triplet(
                str(pre),
                str(post),
                str(lbl)
            )

            n = tile_and_save(
                stacked,
                label,
                sid,
                tiles_dir,
                labels_dir,
                tile_size=TILE_SIZE,
                overlap=OVERLAP,
                min_damage=MIN_DAMAGE
            )

            total_saved += n

        except Exception as e:

            print(f'ERROR: {sid}')
            print(e)

    print(f'{split_name}: {total_saved} new tiles saved')


# ─────────────────────────────────────────────────────────────
# Run preprocessing
# ─────────────────────────────────────────────────────────────
# Remove existing processed data to ensure a clean re-run
import shutil
if Path(PROC_DIR + '/train').exists():
    shutil.rmtree(PROC_DIR + '/train')
if Path(PROC_DIR + '/val').exists():
    shutil.rmtree(PROC_DIR + '/val')

preprocess_split(train_triplets, 'train')

preprocess_split(val_triplets, 'val')

Preprocessing train:   0%|          | 0/2890 [00:00<?, ?it/s]

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — DATASET CLASS
# ══════════════════════════════════════════════════════════════

class BRIGHTDataset(Dataset):

    def __init__(
        self,
        proc_dir,
        split,
        augment=False
    ):

        self.tiles_dir = Path(proc_dir) / split / 'tiles'

        self.labels_dir = Path(proc_dir) / split / 'labels'

        self.tile_files = sorted(
            self.tiles_dir.glob('*.npy')
        )

        self.label_files = sorted(
            self.labels_dir.glob('*.npy')
        )

        self.augment = augment

        # ─────────────────────────────────────────────────────
        # Augmentations
        # ─────────────────────────────────────────────────────
        self.aug = A.Compose([

            A.HorizontalFlip(p=0.5),

            A.VerticalFlip(p=0.5),

            A.RandomRotate90(p=0.5),

        ]) if augment else None

        print(
            f'{split}: '
            f'{len(self.tile_files)} samples'
        )

    def __len__(self):

        return len(self.tile_files)

    def __getitem__(self, idx):

        tile = np.load(
            self.tile_files[idx]
        ).astype(np.float32)

        label = np.load(
            self.label_files[idx]
        ).astype(np.int64)

        if self.aug:

            out = self.aug(
                image=tile,
                mask=label
            )

            tile = out['image']

            label = out['mask']

        tile = torch.from_numpy(
            tile.transpose(2, 0, 1)
        ).float()

        label = torch.from_numpy(
            label
        ).long()

        return tile, label

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12 — DATALOADERS
# ══════════════════════════════════════════════════════════════

train_ds = BRIGHTDataset(
    PROC_DIR,
    'train',
    augment=True
)

val_ds = BRIGHTDataset(
    PROC_DIR,
    'val',
    augment=False
)

print('\nDataset summary:')
print(f'Train: {len(train_ds)}')
print(f'Val:   {len(val_ds)}')


# ─────────────────────────────────────────────────────────────
# Train Loader
# ─────────────────────────────────────────────────────────────
train_loader = DataLoader(

    train_ds,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    drop_last=True
)


# ─────────────────────────────────────────────────────────────
# Validation Loader
# ─────────────────────────────────────────────────────────────
val_loader = DataLoader(

    val_ds,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True
)


# ─────────────────────────────────────────────────────────────
# Verify batch shapes
# ─────────────────────────────────────────────────────────────
imgs, lbls = next(iter(train_loader))

print('\nBatch verification:')
print('Images:', imgs.shape)
print('Labels:', lbls.shape)

print('\nUnique labels:')
print(torch.unique(lbls))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13 — LOSS + METRICS
# ══════════════════════════════════════════════════════════════

CLASS_NAMES = [
    'no_damage',
    'minor',
    'major',
    'destroyed'
]


# ─────────────────────────────────────────────────────────────
# Dice Loss
# ─────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):

        super().__init__()

        self.smooth = smooth

    def forward(self, logits, targets):

        B, C = logits.shape[:2]

        probs = torch.softmax(
            logits,
            dim=1
        )

        one_hot = F.one_hot(
            targets,
            num_classes=C
        ).permute(0,3,1,2).float()

        probs = probs.view(B, C, -1)

        one_hot = one_hot.view(B, C, -1)

        inter = (probs * one_hot).sum(2)

        dice = (
            2 * inter + self.smooth
        ) / (
            probs.sum(2) +
            one_hot.sum(2) +
            self.smooth
        )

        return 1 - dice.mean()


# ─────────────────────────────────────────────────────────────
# Combined Loss
# ─────────────────────────────────────────────────────────────
class CombinedLoss(nn.Module):

    def __init__(self):

        super().__init__()

        self.ce = nn.CrossEntropyLoss()

        self.dice = DiceLoss()

    def forward(self, logits, targets):

        return (
            0.5 * self.ce(logits, targets)
            +
            0.5 * self.dice(logits, targets)
        )


loss_fn = CombinedLoss()


# ─────────────────────────────────────────────────────────────
# mIoU Metric
# ─────────────────────────────────────────────────────────────
def compute_miou(logits, targets):

    preds = logits.argmax(1)

    ious = []

    for c in range(NUM_CLASSES):

        p = preds == c

        t = targets == c

        inter = (p & t).sum().float()

        union = (p | t).sum().float()

        iou = inter / (union + 1e-8)

        ious.append(iou)

    return torch.stack(ious).mean().item()

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 14 — SINGLE MODEL CLASS
# ══════════════════════════════════════════════════════════════

class SegmentationModel(nn.Module):

    def __init__(
        self,
        architecture='unet',
        in_channels=9,
        num_classes=4
    ):

        super().__init__()

        # ─────────────────────────────────────────────────────
        # U-Net
        # ─────────────────────────────────────────────────────
        if architecture == 'unet':

            self.model = smp.Unet(

                encoder_name='resnet50',

                encoder_weights='imagenet',

                in_channels=in_channels,

                classes=num_classes,

                activation=None
            )

        # ─────────────────────────────────────────────────────
        # HRNet
        # ─────────────────────────────────────────────────────
        elif architecture == 'hrnet':

            self.model = smp.Unet(

                encoder_name='tu-hrnet_w32',

                encoder_weights='imagenet',

                in_channels=in_channels,

                classes=num_classes,

                activation=None
            )

        else:
            raise ValueError(
                f'Unknown architecture: {architecture}'
            )

    def forward(self, x):

        return self.model(x)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 15 — TRAINING FUNCTIONS
# ══════════════════════════════════════════════════════════════

loss_fn = CombinedLoss()


# ─────────────────────────────────────────────────────────────
# TRAIN ONE EPOCH
# ─────────────────────────────────────────────────────────────
def train_one_epoch(

    model,

    loader,

    optimizer,

    scaler

):

    model.train()

    total_loss = 0

    for imgs, lbls in tqdm(
        loader,
        desc='Training',
        leave=False
    ):

        imgs = imgs.to(DEVICE)

        lbls = lbls.to(DEVICE)

        optimizer.zero_grad()

        # ─────────────────────────────────────────────────────
        # Mixed Precision
        # ─────────────────────────────────────────────────────
        with torch.cuda.amp.autocast():

            logits = model(imgs)

            loss = loss_fn(
                logits,
                lbls
            )

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


# ─────────────────────────────────────────────────────────────
# VALIDATION
# ─────────────────────────────────────────────────────────────
@torch.no_grad()

def validate(model, loader):

    model.eval()

    total_loss = 0

    total_iou = 0

    for imgs, lbls in tqdm(
        loader,
        desc='Validation',
        leave=False
    ):

        imgs = imgs.to(DEVICE)

        lbls = lbls.to(DEVICE)

        logits = model(imgs)

        loss = loss_fn(
            logits,
            lbls
        )

        miou = compute_miou(
            logits,
            lbls
        )

        total_loss += loss.item()

        total_iou += miou

    return (

        total_loss / len(loader),

        total_iou / len(loader)
    )

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 16 — TRAIN MODEL WITH RESUME
# ══════════════════════════════════════════════════════════════

def train_model(

    model,

    name,

    train_loader,

    val_loader,

    epochs=50,

    lr=1e-4,

    patience=10

):

    model = model.to(DEVICE)

    # ─────────────────────────────────────────────────────────
    # AMP scaler
    # ─────────────────────────────────────────────────────────
    scaler = torch.cuda.amp.GradScaler()

    # ─────────────────────────────────────────────────────────
    # Optimizer
    # ─────────────────────────────────────────────────────────
    optimizer = optim.AdamW(

        model.parameters(),

        lr=lr,

        weight_decay=1e-4
    )

    # ─────────────────────────────────────────────────────────
    # Scheduler
    # FIXED verbose=True issue
    # ─────────────────────────────────────────────────────────
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode='max',

        factor=0.5,

        patience=3
    )

    # ─────────────────────────────────────────────────────────
    # Checkpoints
    # ─────────────────────────────────────────────────────────
    ckpt_last = f'{CHECKPOINT_DIR}/{name}_last.pth'

    ckpt_best = f'{CHECKPOINT_DIR}/{name}_best.pth'

    # ─────────────────────────────────────────────────────────
    # Resume Variables
    # ─────────────────────────────────────────────────────────
    best_iou = 0

    start_epoch = 1

    patience_counter = 0

    # ─────────────────────────────────────────────────────────
    # Resume Training
    # ─────────────────────────────────────────────────────────
    if os.path.exists(ckpt_last):

        print(f'Resuming from checkpoint: {ckpt_last}')

        ckpt = torch.load(

            ckpt_last,

            map_location=DEVICE
        )

        model.load_state_dict(
            ckpt['model_state']
        )

        optimizer.load_state_dict(
            ckpt['optimizer_state']
        )

        scaler.load_state_dict(
            ckpt['scaler_state']
        )

        best_iou = ckpt['best_iou']

        start_epoch = ckpt['epoch'] + 1

        patience_counter = ckpt['patience_counter']

        print(f'Resumed from epoch {start_epoch}')

        print(f'Best mIoU: {best_iou:.4f}')

    # ─────────────────────────────────────────────────────────
    # Training Loop
    # ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, epochs + 1):

        print('\n' + '='*60)

        print(f'EPOCH {epoch}/{epochs}')

        print('='*60)

        train_loss = train_one_epoch(

            model,

            train_loader,

            optimizer,

            scaler
        )

        val_loss, val_iou = validate(

            model,

            val_loader
        )

        scheduler.step(val_iou)

        current_lr = optimizer.param_groups[0]['lr']

        print(
            f'Train Loss : {train_loss:.4f}'
        )

        print(
            f'Val Loss   : {val_loss:.4f}'
        )

        print(
            f'Val mIoU   : {val_iou:.4f}'
        )

        print(
            f'LR          : {current_lr:.2e}'
        )

        # ─────────────────────────────────────────────────────
        # Save latest checkpoint EVERY epoch
        # ─────────────────────────────────────────────────────
        save_dict = {

            'epoch': epoch,

            'model_state': model.state_dict(),

            'optimizer_state': optimizer.state_dict(),

            'scaler_state': scaler.state_dict(),

            'best_iou': best_iou,

            'patience_counter': patience_counter
        }

        torch.save(
            save_dict,
            ckpt_last
        )

        # ─────────────────────────────────────────────────────
        # Save BEST model
        # ─────────────────────────────────────────────────────
        if val_iou > best_iou:

            best_iou = val_iou

            patience_counter = 0

            save_dict['best_iou'] = best_iou

            torch.save(
                save_dict,
                ckpt_best
            )

            print(
                f'✅ BEST MODEL SAVED '
                f'(mIoU={best_iou:.4f})'
            )

        else:

            patience_counter += 1

            print(
                f'Patience: '
                f'{patience_counter}/{patience}'
            )

        # ─────────────────────────────────────────────────────
        # Early Stopping
        # ─────────────────────────────────────────────────────
        if patience_counter >= patience:

            print('\nEARLY STOPPING TRIGGERED')

            break

        # ─────────────────────────────────────────────────────
        # Free cache
        # ─────────────────────────────────────────────────────
        torch.cuda.empty_cache()

        gc.collect()

    print('\nTraining Complete')

    print(f'Best mIoU: {best_iou:.4f}')

    print(f'Best checkpoint: {ckpt_best}')

    return model

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 17 — TRAIN U-NET
# ══════════════════════════════════════════════════════════════

unet = SegmentationModel(
    architecture='unet'
)

unet = train_model(

    model=unet,

    name='unet',

    train_loader=train_loader,

    val_loader=val_loader,

    epochs=50,

    lr=1e-4,

    patience=10
)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 18 — CLEAR VRAM
# ══════════════════════════════════════════════════════════════

del unet

torch.cuda.empty_cache()

gc.collect()

print('VRAM cleared')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 19 — TRAIN HRNET
# ══════════════════════════════════════════════════════════════

hrnet = SegmentationModel(
    architecture='hrnet'
)

hrnet = train_model(

    model=hrnet,

    name='hrnet',

    train_loader=train_loader,

    val_loader=val_loader,

    epochs=50,

    lr=1e-4,

    patience=10
)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 20 — LOAD BEST MODEL
# ══════════════════════════════════════════════════════════════

best_model = SegmentationModel(
    architecture='hrnet'
).to(DEVICE)

ckpt = torch.load(

    f'{CHECKPOINT_DIR}/hrnet_best.pth',

    map_location=DEVICE
)

best_model.load_state_dict(
    ckpt['model_state']
)

best_model.eval()

print('Best HRNet model loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 21 — VISUALIZE PREDICTIONS
# ══════════════════════════════════════════════════════════════

DAMAGE_CMAP = plt.matplotlib.colors.ListedColormap(

    [
        '#2ecc71',
        '#f1c40f',
        '#e67e22',
        '#e74c3c'
    ]
)

fig, axes = plt.subplots(
    4,
    3,
    figsize=(15, 20)
)

with torch.no_grad():

    for i in range(4):

        tile, lbl = val_ds[i]

        pred = best_model(

            tile.unsqueeze(0).to(DEVICE)

        )

        pred = pred.argmax(1)

        pred = pred.squeeze().cpu().numpy()

        rgb = tile[:3].permute(1,2,0).numpy()

        rgb = (

            rgb - rgb.min()

        ) / (

            rgb.max() - rgb.min() + 1e-8
        )

        # RGB
        axes[i][0].imshow(rgb)

        axes[i][0].set_title('RGB')

        # GT
        axes[i][1].imshow(
            lbl.numpy(),
            cmap=DAMAGE_CMAP
        )

        axes[i][1].set_title('Ground Truth')

        # Prediction
        axes[i][2].imshow(
            pred,
            cmap=DAMAGE_CMAP
        )

        axes[i][2].set_title('Prediction')

        for ax in axes[i]:

            ax.axis('off')

plt.tight_layout()

plt.show()